<a href="https://colab.research.google.com/github/Ajjme/Powergrid_optimization/blob/main/Powergrid_Optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import numpy as np

In [8]:
# LP analysis from attachment in email
def LP_analysis(x, constants):
    """
    [f, g] = LP_analysis(x, constants)
    Analyze a trial solution x to a linear programming problem:
      minimize f = c^T x  such that  g = A x - b <= 0

    Parameters
    ----------
    x : array_like
        Decision vector (n,)
    constants : sequence
        constants[0] -> A : ndarray (m, n)  constraint coefficient matrix
        constants[1] -> b : ndarray (m,)    constraint vector
        constants[2] -> c : ndarray (n,)    cost coefficient vector

    Returns
    -------
    f : float
        The scalar cost c^T x
    g : ndarray
        Constraint inequalities A x - b  (m,)
    """
    # Convert inputs to numpy arrays (defensive)
    x = np.asarray(x).ravel()
    A = np.asarray(constants[0])
    b = np.asarray(constants[1]).ravel()
    c = np.asarray(constants[2]).ravel()

    # Validate shapes (helpful errors if shapes mismatch)
    if A.shape[1] != x.size:
        raise ValueError(f"Incompatible shapes: A has {A.shape[1]} cols but x has length {x.size}")
    if A.shape[0] != b.size:
        raise ValueError(f"Incompatible shapes: A has {A.shape[0]} rows but b has length {b.size}")
    if c.size != x.size:
        raise ValueError(f"Incompatible shapes: c has length {c.size} but x has length {x.size}")

    # compute cost
    f = float(np.dot(c, x))   # scalar

    # compute constraint inequalities
    g = A.dot(x) - b          # vector of length m

    return f, g


In [9]:
# 13 line version fro PDF
def LP_analysis(x, constants):
    # [f, g] = LP_analysis(x, constants)
    # analyze a trial solution x to a linear programming problem,
    # minimize f = c' * x such that g = A * x - b <= 0

    A = constants[0]   # constraint coefficient matrix (dimension m by n)
    b = constants[1]   # constraint vector (dimension m by 1)
    c = constants[2]   # cost coefficient vector (dimension n by 1)

    f = np.dot(c.T, x)  # the cost function

    g = A @ x - b       # the constraint inequalities, compared to zero

    return f, g


In [15]:
# opt_options.py
# -----------------------------------------------------------------------------
# default optimization parameters for Multivarious optimization routines
# -----------------------------------------------------------------------------

def opt_options(options_in=None):
    """
    Return the default optimization parameters, optionally updating with user input.

    Parameters
    ----------
    options_in : list or np.ndarray, optional
        Custom user-defined options. Missing or zero values will be replaced by defaults.

    Returns
    -------
    options : np.ndarray
        Vector of optimization settings.
    """

    # Default parameters (same order and values as MATLAB)
    default_options = np.array([
        1,       # (0) message level flag
        1e-3,    # (1) tolerance on parameters
        1e-3,    # (2) tolerance on cost
        0e-4,    # (3) tolerance on constraints
        1000,    # (4) max number of function evaluations
        10,      # (5) penalty on constraint violations
        1,       # (6) exponent on constraint violations
        1,       # (7) number of function evaluations in average
        0.1,     # (8) desired coefficient of variation on mean estimate
        0,       # (9) stop when feasible
        0,       # (10) index for plotting surface
        1,       # (11) index for plotting surface
        25,      # (12) # of first-index values for plotting
        35,      # (13) # of second-index values for plotting
        1e-6,    # (14) finite diff minimum param. change
        2,       # (15) penalty type
        1e-6,    # (17) min param. change for finite diff gradients
        1e-1,    # (17) max param. change for finite diff gradients
        0        # (18) number of equality constraints
    ], dtype=float)

    # Initialize
    if options_in is None:
        return default_options.copy()

    options_in = np.abs(np.array(options_in, dtype=float))
    n = len(options_in)
    options = default_options.copy()
    options[:n] = options_in[:n]

    # Sanity checks and constraints
    if n > 5 and options_in[5] == 0:
        options[5] = 0
    options[0] = abs(round(options[0]))
    options[6] = np.clip(options[6], 0.001, 5)
    options[7] = max(options[7], 1)
    options[8] = abs(options[8])
    options[9] = abs(options[9])

    return options

In [13]:
"""
plot_opt_surface.py - 3D Surface Plot of Objective Function

Draw a surface plot of J(x) vs. x(i), x(j), where all other values in x
are held constant. Useful for visualizing optimization landscapes and
convergence paths.

Translation from MATLAB to Python by Claude, 2025-11-19
Original by H.P. Gavin, Civil & Environ. Eng'g, Duke Univ.
Updated 2015-09-29, 2016-03-23, 2016-09-12, 2020-01-20, 2021-12-31, 2025-01-26
"""

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D


def plot_opt_surface(func, x, v_lb, v_ub, options, consts=None, fig_no=1):
    """
    Draw a surface plot of objective function J(x) vs. x(i), x(j).

    Creates a 3D mesh surface showing the objective function landscape over
    two design variables while holding all others constant. Marks the initial
    point and the grid minimum on the surface.

    Parameters
    ----------
    func : callable
        Function to be optimized with signature:
        [objective, constraints] = func(x, consts)
        Returns objective value (float) and constraint values (array)
    x : ndarray, shape (n,)
        Vector of initial parameter values (column vector)
    v_lb : ndarray, shape (n,)
        Lower bounds on permissible parameter values
    v_ub : ndarray, shape (n,)
        Upper bounds on permissible parameter values
    options : ndarray or list
        options[0] = 3 (reserved for surface plotting flag)
        options[3] = tol_g - tolerance on constraint functions
        options[5] = penalty - penalty factor on constraint violations
        options[6] = q - exponent on constraint violations
        options[10] = i - 1st index for plotting (0-based in Python!)
        options[11] = j - 2nd index for plotting (0-based in Python!)
        options[12] = Ni - number of points in 1st dimension
        options[13] = Nj - number of points in 2nd dimension
    consts : optional
        Optional vector of constants to be passed to func(x, consts)
    fig_no : int, optional
        Figure number for the plot (default: 1)

    Returns
    -------
    fmin : float
        Minimum value of the meshed surface data
    fmax : float
        Maximum value of the meshed surface data (clipped for visualization)
    ax :
        To allow other functions to update the figure

    Notes
    -----
    - Handles constraint violations by adding penalties to objective
    - Automatically scales z-axis to avoid extreme outliers
    - Marks initial point (green) and grid minimum (red) on surface
    - If penalty <= 0 and constraints violated, point is set to NaN
    """

    # Convert inputs to numpy arrays
    x = np.asarray(x).flatten()
    v_lb = np.asarray(v_lb).flatten()
    v_ub = np.asarray(v_ub).flatten()
    options = np.asarray(options)

    # Extract options (note: MATLAB is 1-indexed, Python is 0-indexed)
    tol_g = options[3]      # Constraint tolerance
    penalty = options[5]    # Penalty factor
    q = options[6]          # Penalty exponent
    i = int(options[10])    # 1st variable index (0-based)
    j = int(options[11])    # 2nd variable index (0-based)
    Ni = int(options[12])   # Number of points in 1st dimension
    Nj = int(options[13])   # Number of points in 2nd dimension

    # Store initial x values
    v_init = x.copy()

    # Create grid for the two variables
    v_i = np.linspace(v_lb[i], v_ub[i], Ni)
    v_j = np.linspace(v_lb[j], v_ub[j], Nj)

    # Initialize objective function mesh
    f_mesh = np.full((Ni, Nj), np.nan)

    # Evaluate objective function on grid
    for ii in range(Ni):
        for jj in range(Nj):
            # Set the two variables to grid values
            x[i] = v_i[ii]
            x[j] = v_j[jj]

            # Evaluate function
            if consts is not None:
                f, g = func(x, consts)
            else:
                f, g = func(x)

            # Ensure g is an array
            g = np.asarray(g).flatten()

            # Add penalty for constraint violations
            constraint_penalty = np.sum(g * (g > tol_g)) ** q
            f_mesh[ii, jj] = f + penalty * constraint_penalty

            # If no penalty and constraints violated, set to NaN
            if penalty <= 0 and np.max(g) > tol_g:
                f_mesh[ii, jj] = np.nan

    # Compute RMS and set z-axis limits
    frms = np.sqrt(np.nansum(f_mesh**2))
    fmin = 0.9 * np.nanmin(f_mesh)
    fmax = min(np.nanmax(f_mesh), 5 * frms)

    # Handle case where all values are NaN
    if np.isnan(fmin) or np.isnan(fmax):
        fmin = 0.0
        fmax = 1.0

    # Find grid minimum location
    if np.all(np.isnan(f_mesh)):
        ii_min, jj_min = 0, 0
    else:
        min_val = np.nanmin(f_mesh)
        ii_min, jj_min = np.where(f_mesh == min_val)
        ii_min = ii_min[0]  # Take first if multiple minima
        jj_min = jj_min[0]

    # Create the plot
    plt.ion() # interactive plot mode: on
    fig = plt.figure(fig_no, figsize=(12, 9))
    fig.clf()
    ax = fig.add_subplot(111, projection='3d')

    # Create meshgrid for plotting
    X_i, X_j = np.meshgrid(v_i, v_j, indexing='ij')

    # Plot the surface mesh
    # ax.plot_wireframe(X_i, X_j, f_mesh, linewidth=1.5, alpha=0.7)

    # Alternative: use plot_surface for filled surface
    ax.plot_surface(X_i, X_j, f_mesh, cmap='viridis', alpha=0.5, \
                    linewidth=0.5, edgecolor='k')

    # Set labels with LaTeX-style formatting
    ax.set_xlabel(f'$v_{{{i+1}}}$', fontsize=14)
    ax.set_ylabel(f'$v_{{{j+1}}}$', fontsize=14)
    ax.set_zlabel(f'objective   $f_A(v_{{{i+1}}}, v_{{{j+1}}})$', fontsize=14)

    # Set axis limits
    ax.set_xlim([np.min(v_i), np.max(v_i)])
    ax.set_ylim([np.min(v_j), np.max(v_j)])
    ax.set_zlim([fmin, fmax])

    # Plot initial point (green circle)
    v_plot = v_init.copy()
    if consts is not None:
        f_init, g_init = func(v_init, consts)
    else:
        f_init, g_init = func(v_init)

    g_init = np.asarray(g_init).flatten()
    constraint_penalty_init = np.sum(g_init * (g_init > tol_g)) ** q
    fa_init = f_init + penalty * constraint_penalty_init

    # Offset points slightly above surface for visibility
    z_offset = (fmax - fmin) / 50

    ax.plot([v_init[i]], [v_init[j]], [fa_init + z_offset],
            'o', alpha=1.0, color='green', markersize=22, markeredgewidth=3,
            markerfacecolor='green', markeredgecolor='darkgreen',
            label='Initial point')

    # Plot grid minimum (red circle)
    ax.plot([v_i[ii_min]], [v_j[jj_min]], [f_mesh[ii_min, jj_min] + z_offset],
            'o', alpha=1.0, color='red', markersize=22, markeredgewidth=3,
            markerfacecolor='red', markeredgecolor='darkred',
            label='Grid minimum')

    # Add legend
    #ax.legend(fontsize=11)

    # Improve viewing angle
    ax.view_init(elev=25, azim=-60)

    # Add grid
    ax.grid(True, alpha=0.3)

    # Tight layout
    plt.tight_layout()

    # Save figure
    #filename = f'plot_opt_surface-{fig_no}.png'
    #plt.savefig(f'/mnt/user-data/outputs/{filename}', dpi=150, bbov_inches='tight')
    #print(f"Saved: {filename}")

    return fmin, fmax, ax


# Example usage / test
if __name__ == "__main__":
    print("\n" + "="*70)
    print("Testing plot_opt_surface.py")
    print("="*70 + "\n")

    # Define a simple test function (Rosenbrock function)
    def test_func(x, consts=None):
        """
        Rosenbrock function: f(x) = sum((1-v_i)^2 + 100*(v_{i+1} - v_i^2)^2)
        Minimum at x = [1, 1, ...] with f(x) = 0

        Returns
        -------
        f : float
            Objective function value
        g : ndarray
            Constraint values (no constraints in this example)
        """
        x = np.asarray(x).flatten()
        n = len(x)

        # Rosenbrock function
        f = 0.0
        for i in range(n - 1):
            f += (1 - x[i])**2 + 100 * (x[i+1] - x[i]**2)**2

        # No constraints (return array of negative values = feasible)
        g = np.array([-1.0])

        return f, g

    # Set up parameters
    n = 3  # Number of design variables
    v_init = np.array([0.0, 0.0, 0.0])  # Initial point
    v_lb = np.array([-2.0, -2.0, -2.0])  # Lower bounds
    v_ub = np.array([2.0, 2.0, 2.0])     # Upper bounds

    # Set up options array
    options = np.zeros(14)
    options[0] = 3        # Surface plot flag
    options[3] = 0.0      # Constraint tolerance
    options[5] = 0.0      # Penalty (no penalty needed - no constraints)
    options[6] = 2.0      # Penalty exponent
    options[10] = 0       # Plot v_0 (1st variable, 0-indexed)
    options[11] = 1       # Plot v_1 (2nd variable, 0-indexed)
    options[12] = 30      # Number of points in v_0 direction
    options[13] = 30      # Number of points in v_1 direction

    # Create the surface plot
    # print("Creating 3D surface plot of Rosenbrock function...")
    # print(f"Initial point: x = {v_init}")
    # print(f"Plotting x[{int(options[10])}] vs x[{int(options[11])}]")
    # print(f"Grid size: {int(options[12])} x {int(options[13])}")

    # fmin, fmax = plot_opt_surface(test_func, v_init, v_lb, v_ub,
    #                                options, consts=None, fig_no=100)

    # print(f"\nSurface z-axis range: [{fmin:.4f}, {fmax:.4f}]")

    # plt.show()

    # print("\n" + "="*70)
    # print("plot_opt_surface test completed successfully!")
    # print("Figure saved to /mnt/user-data/outputs/")
    # print("="*70 + "\n")


Testing plot_opt_surface.py



In [25]:
# fucntion from Herni's Github

# sqp.py
# -----------------------------------------------------------------------------
# Sequential Quadratic Programming for Nonlinear Optimization
# Translation of Henri P. Gavin's SQPopt.m (Duke CEE).
# Depends on: opt_options(), plot_opt_surface()
# Uses quadprog package for QP subproblems
# -----------------------------------------------------------------------------

from __future__ import annotations
import time
import numpy as np

from scipy.optimize import minimize
from scipy.linalg import cho_factor, cho_solve

#from opt_options import opt_options
from matplotlib import pyplot as plt
#from plot_opt_surface import plot_opt_surface
#from quadprog import solve_qp


def sqp(func, v_init, v_lb=None, v_ub=None, options_in=None, consts=1.0):
    """
    Sequential Quadratic Programming for nonlinear optimization with inequality constraints.

    Minimizes f(v) such that g(v) < 0 and v_lb <= v_opt <= v_ub.
    Uses finite differences for gradients and BFGS Hessian updates.

    Parameters
    ----------
    func : callable
        Signature: f, g = func(v, consts).  f is scalar, g is (m,) constraints (g<0 feasible).
        v is in *original* units (not scaled).
    v_init : array-like (n,)
        Initial guess.
    v_lb, v_ub : array-like (n,), optional
        Lower/upper bounds on v. If omitted, wide bounds are used (±1e2*|v_init|).
    options_in : array-like, optional
        See opt_options() for the 19 parameters (same positions as MATLAB).
    consts : any
        Passed through to `func`.

    Returns
    -------
    v_opt : np.ndarray (n,)
        Optimal design variables
    f_opt : float
        Optimal objective value
    g_opt : np.ndarray (m,)
        Optimal constraint values
    cvg_hst : np.ndarray (n+5, k)
        Convergence history: [v; f; max(g); func_count; cvg_v; cvg_f] per iteration
    lambda_opt : np.ndarray
        Lagrange multipliers at active constraints
    hess : np.ndarray (n, n)
        Final Hessian matrix approximation

    References
    ----------
    Copyright (c) 1990 by The MathWorks, Inc., Andy Grace 7-9-90.
    Modified and enhanced by H.P. Gavin, Duke University.
    """

    # ----- options & inputs -----
    v_init = np.asarray(v_init, dtype=float).flatten()
    n = v_init.size

    if v_lb is None or v_ub is None:
        v_lb = -1.0e2 * np.abs(v_init)
        v_ub = +1.0e2 * np.abs(v_init)
    v_lb = np.asarray(v_lb, dtype=float).flatten()
    v_ub = np.asarray(v_ub, dtype=float).flatten()

    # Check for valid bounds
    if np.any(v_ub <= v_lb):
        raise ValueError("v_ub must be greater than v_lb for all parameters")

    # Put initial guess within bounds
    v_init = np.clip(v_init, 0.9 * v_lb, 0.9 * v_ub)

    options = opt_options(options_in)
    msglev    = int(options[0])    # display level
    tol_v     = float(options[1])  # design var convergence tol
    tol_f     = float(options[2])  # objective convergence tol
    tol_g     = float(options[3])  # constraint tol
    max_evals = int(options[4])    # budget
    del_min   = float(options[16]) # min parameter change for finite diff
    del_max   = float(options[17]) # max parameter change for finite diff
    options[5] = -1                # no penalty factor involved in SQP

    # ----- scale to [-1, +1] -----
    s0 = (v_lb + v_ub) / (v_lb - v_ub)
    s1 = 2.0 / (v_ub - v_lb)
    x = s0 + s1 * v_init
    x_lb = -1.0 * np.ones(n)
    x_ub = +1.0 * np.ones(n)

    # ----- initialize -----
    function_count = 0
    iteration = 1
        # AJ add: initialize convergence metrics so they exist before first use
    cvg_v = 0.0
    cvg_f = 0.0
    ###################
    cvg_hst = np.full((n + 5, max(1, max_evals)), np.nan)

    t0 = time.time()

    # First function evaluation
    f, g = func((x - s0) / s1, consts)
    function_count += 1

    if not np.isscalar(f):
        raise ValueError("Objective returned by func(v,consts) must be a scalar.")
    g = np.atleast_1d(g).astype(float).flatten()
    m = g.size  # number of constraints

    # Initialize best solution
    f_opt = float(f)
    x_opt = x.copy()
    g_opt = g.copy()

    # Save initial state
    cvg_hst[:, iteration - 1] = np.concatenate([(x - s0) / s1,
                                                [f, np.max(g), function_count, 1.0, 1.0]])

    if msglev > 2:
        f_min, f_max, ax = plot_opt_surface(func, (x-s0)/s1, v_lb, v_ub, options, consts, 103)

    # Initialize gradient and Hessian storage
    OLDX = x.copy()
    OLDG = g.copy()
    gradf = np.zeros(n)
    OLDgradf = np.zeros(n)
    gradg = np.zeros((m, n))
    OLDgradg = np.zeros((m, n))
    LAMBDA = np.zeros(m)  # Lagrange multipliers
    HESS = np.eye(n)      # Hessian approximation
    PENALTY = np.ones(m)  # Penalty factors

    # Finite difference step sizes
    CHG = 1e-7 * (1.0 + np.abs(x))
    GNEW = 1e8 * CHG

    StepLength = 1.0
    end_iterations = False

    if msglev:
        print(" *********************** SQP ****************************")
        print(f" iteration                = {iteration:5d}   "
              f"{'*** feasible ***' if np.max(g) <= tol_g else '!!! infeasible !!!'}")
        print(f" function evaluations     = {function_count:5d} of {max_evals:5d}")
        print(f" objective                = {f:11.3e}")
        print(" variables                = " + " ".join(f"{v:11.3e}" for v in (x-s0)/s1))
        print(f" max constraint           = {np.max(g):11.3e}")
        print("\n")

    # ============================ main loop ============================
    while not end_iterations:

        # ----- Compute gradients via finite differences -----
        oldf = f
        oldg = g.copy()

        # Adaptive step sizing
        CHG = -1.0e-8 / (GNEW + np.finfo(float).eps)
        CHG = np.sign(CHG + np.finfo(float).eps) * np.clip(np.abs(CHG), del_min, del_max)

        for i in range(n):
            temp = x[i]
            x[i] = temp + CHG[i]
            f_fd, g_fd = func((x - s0) / s1, consts)
            g_fd = np.atleast_1d(g_fd).astype(float).flatten()  # Ensure proper shape
            function_count += 1

            # Update best solution if improved
            if np.max(g_fd) < tol_g and f_fd < f_opt:
                if msglev > 1:
                    print(' update optimum point')
                f_opt = f_fd
                g_opt = g_fd.copy()
                x_opt = x.copy()

            gradf[i] = (f_fd - oldf) / CHG[i]
            gradg[:, i] = (g_fd - oldg) / CHG[i]
            x[i] = temp

        f = oldf
        g = oldg.copy()

        # Initialize penalty on first iteration
        if iteration == 1:
            PENALTY = (np.finfo(float).eps + np.dot(gradf, gradf)) * np.ones(m) / \
                     (np.sum(gradg**2, axis=1) + np.finfo(float).eps)

        # Compute gradient of augmented Lagrangian
        GOLD = OLDgradf + np.dot(LAMBDA, OLDgradg)
        GNEW = gradf + np.dot(LAMBDA, gradg)
        q = GNEW - GOLD  # change in augmented gradient
        p = x - OLDX     # change in design variables

        # ----- Update Hessian (BFGS) -----
        how = 'regular'
        qp = np.dot(q, p)

        # Ensure Hessian positive definiteness
        if qp < StepLength**2 * 1e-3:
            how = 'modify gradients to ensure Hessian > 0'
            # Modify q to ensure qp > 0
            while qp < -1e-5:
                qp_components = q * p
                idx_min = np.argmin(qp_components)
                q[idx_min] = q[idx_min] / 2
                qp = np.dot(q, p)

            if qp < np.finfo(float).eps * np.linalg.norm(HESS, 'fro'):
                FACTOR = np.dot(gradg.T, g) - np.dot(OLDgradg.T, OLDG)
                FACTOR = FACTOR * (p * FACTOR > 0) * (q * p <= np.finfo(float).eps)
                WT = 1e-2
                if np.max(np.abs(FACTOR)) == 0:
                    FACTOR = 1e-5 * np.sign(p)
                    how = 'small gradients'
                while qp < np.finfo(float).eps * np.linalg.norm(HESS, 'fro') and WT < 1 / np.finfo(float).eps:
                    q = q + WT * FACTOR
                    qp = np.dot(q, p)
                    WT = WT * 2

        # Perform BFGS update if qp > 0
        if qp > np.finfo(float).eps:
            HESS = HESS + np.outer(q, q) / qp - \
                   np.dot(HESS, np.outer(p, p)).dot(HESS) / np.dot(p, HESS).dot(p)
            how = how + ', Hessian update'
        else:
            how = how + ', no Hessian update'

        # ----- Save old values -----
        OLDX = x.copy()
        OLDF = f
        OLDG = g.copy()
        OLDgradf = gradf.copy()
        OLDgradg = gradg.copy()
        OLDLAMBDA = LAMBDA.copy()

        # ----- Solve QP subproblem for search direction -----
        # Append box constraints to nonlinear constraints
        GT = np.concatenate([g, -x + x_lb, x - x_ub])
        gradg_augmented = np.vstack([gradg, -np.eye(n), np.eye(n)])

        # Solve QP: min 0.5*SD'*HESS*SD + gradf'*SD  s.t.  gradg_augmented*SD <= -GT
        SD, lambda_qp, howqp = solve_qp_subproblem(HESS, gradf, gradg_augmented, -GT)

        # Extract Lagrange multipliers for nonlinear constraints only
        LAMBDA = lambda_qp[:m]

        # Update penalty factors (don't change too quickly)
        PENALTY = np.maximum(LAMBDA, 0.5 * (LAMBDA + PENALTY))

        g_max = np.max(g)

        # ----- Line search -----
        infeas = (howqp == 'infeasible')

        # Goal functions for merit-based line search
        GOAL_1 = f + np.sum(PENALTY * (g > tol_g) * g) + 1e-30
        if g_max > tol_g:
            GOAL_2 = g_max
        elif f >= 0:
            GOAL_2 = -1 / (f + 1)
        else:
            GOAL_2 = 0
        if not infeas and f < 0:
            GOAL_2 = GOAL_2 + f - 1

        COST_1 = GOAL_1 + 1
        COST_2 = GOAL_2 + 1

        if msglev > 1:
            print('   alpha      max{g}          COST_1           COST_2')

        StepLength = 2.0

        while (COST_1 > GOAL_1) and (COST_2 > GOAL_2) and (function_count < max_evals):
            StepLength = StepLength / 2
            if StepLength < 1e-4:
                StepLength = -StepLength  # change direction

            x = OLDX + StepLength * SD
            f, g = func((x - s0) / s1, consts)
            g = np.atleast_1d(g).astype(float).flatten()  # Ensure proper shape
            function_count += 1

            # Update best solution
            if np.max(g) < tol_g and f < f_opt:
                if msglev > 1:
                    print(' update optimum')
                f_opt = f
                g_opt = g.copy()
                x_opt = x.copy()

            g_max = np.max(g)
            COST_1 = f + np.sum(PENALTY * (g > 0) * g)
            if g_max > tol_g:
                COST_2 = g_max
            elif f >= 0:
                COST_2 = -1 / (f + 1)
            else:
                COST_2 = 0
            if not infeas and f < 0:
                COST_2 = COST_2 + f - 1

            if msglev > 1:
                print(f'  {StepLength:9.2e}   {g_max:9.2e}  {COST_1:9.2e} '
                      f'{COST_1/GOAL_1:6.3f} {COST_2:9.2e} {COST_2/GOAL_2:6.3f}')

        # ----- Update Lagrange multipliers -----
        absSL = abs(StepLength)
        LAMBDA = absSL * LAMBDA + (1 - absSL) * OLDLAMBDA
        g_ok = g < -tol_g
        LAMBDA[g_ok] = 0
        # Note: lambda_qp includes box constraints, LAMBDA is only nonlinear constraints

        iteration += 1

        # ----- Display progress -----
        if msglev:
            elapsed = time.time() - t0
            rate = function_count / max(elapsed, 1e-9)
            remaining = max_evals - function_count
            eta_sec = int(remaining / max(rate, 1e-9))

            g_max, idx_max_g = np.max(g), np.argmax(g)


            cvg_f = abs(absSL * np.dot(gradf, SD) / f) if f != 0 else 0
            cvg_v = np.max(np.abs(absSL * SD / (x + np.finfo(float).eps)))

            print("\n *********************** SQP ****************************")
            print(f" iteration                = {iteration:5d}   "
                  f"{'*** feasible ***' if g_max <= tol_g and np.all(x >= -1) and np.all(x <= 1) else '!!! infeasible !!!'}")
            print(f" function evaluations     = {function_count:5d} of {max_evals:5d}"
                  f" ({100.0*function_count/max_evals:4.1f}%)")
            print(f" e.t.a.                   = ~{eta_sec//60}m{eta_sec%60:02d}s")
            print(f" objective                = {f:11.3e}")
            print(" variables                = " + " ".join(f"{v:11.3e}" for v in (x-s0)/s1))
            print(f" max constraint           = {g_max:11.3e}  ({idx_max_g+1})")
            print(f" Step Size                = {StepLength:11.3e}")
            print(f" BFGS method              : {how}")
            print(f" QP method                : {howqp}")
            print(f" Convergence F            = {cvg_f:11.4e}   tolF = {tol_f:8.6f}")
            print(f" Convergence X            = {cvg_v:11.4e}   tolX = {tol_v:8.6f}")
            print("\n")

        # Save convergence history
        cvg_hst[:, iteration - 1] = np.concatenate([
            (x - s0) / s1,
            [f, g_max, function_count, cvg_v, cvg_f]
        ])

        # Plot current point
        if msglev > 2:
            ii = int(options[10])
            jj = int(options[11])
            x_plot = (x - s0) / s1
            ax.plot([x_plot[ii]], [x_plot[jj]], f ,
                   'ro', alpha=1.0, markersize=8, linewidth=4,
                   markerfacecolor='red', markeredgecolor='darkred')
            plt.draw()

        # ----- Check convergence -----
        cvg_f = abs(absSL * np.dot(gradf, SD) / f) if f != 0 else 0
        cvg_v = np.max(np.abs(absSL * SD / ( (x-s0)/s1 + 1e-6 )))

        if ((cvg_v < tol_v or cvg_f < tol_f) and
            ((g_max < tol_g) or (howqp == 'infeasible' and g_max > 0))):

            end_iterations = True

            if howqp != 'infeasible':
                print(f' * Woo Hoo!  Converged solution found in {iteration} iterations!')
                if cvg_v < tol_v:
                    print(' *           convergence in design variables')
                if cvg_f < tol_f:
                    print(' *           convergence in design objective')
                if g_max < tol_g:
                    print(' * Woo Hoo!  Converged solution is feasible')
                else:
                    print(' * Boo Hoo!  Converged solution is NOT feasible!')
            else:
                if g_max > tol_g:
                    print(' * Boo Hoo Hoo!  No feasible solution found.')

        elif function_count >= max_evals:
            x_opt = OLDX
            f_opt = OLDF
            g_opt = OLDG
            print(f' * Enough! Maximum number of function evaluations ({max_evals}) exceeded')
            print(' * Increase tol_v (options[1]), tol_f (options[2]), or max_evals (options[4])')
            print(' * and try, try, try again!')
            end_iterations = True

    # ============================ end main loop ============================

    # If better feasible solution was found during search, use it
    if f_opt < f and np.max(g_opt) < tol_g:
        x = x_opt
        f = f_opt
        g = g_opt

    # Scale back to original units
    v_opt = (x - s0) / s1
    f_opt = f
    g_opt = g

    # ----- Summary -----
    if msglev:
        dur = time.time() - t0
        print(f" *          objective = {f_opt:11.3e}   evals = {function_count}   "
              f"time = {dur:.2f}s")
        print(" * ----------------------------------------------------------------------------")
        print(" *                v_init      v_lb     <    v_opt     <    v_ub      lambda")
        print(" * ----------------------------------------------------------------------------")
        for i in range(n):
            eqlb = '=' if v_opt[i] < v_lb[i] + tol_g + 10 * 1e-12 else ' '
            equb = '=' if v_opt[i] > v_ub[i] - tol_g - 10 * 1e-12 else ' '
            lulb = ''
            if eqlb == '=':
                lulb = f'{lambda_qp[m + i]:12.5f}'
            elif equb == '=':
                lulb = f'{lambda_qp[m + n + i]:12.5f}'
            print(f" *  v[{i+1:3d}]  {v_init[i]:11.4f} "
                  f"{v_lb[i]:11.4f} {eqlb} {v_opt[i]:12.5f} {equb} {v_ub[i]:11.4f} {lulb}")
        print(" * ----------------------------------------------------------------------------")
        print(" * Constraints:")
        for j in range(m):
            binding = ''
            if lambda_qp[j] > 0:
                binding = '    ** binding **'
            if g_opt[j] >= tol_g:
                binding = '    ** not ok  **'
            print(f" *  g[{j+1:3d}] = {g_opt[j]:12.5f}      "
                  f"lambda[{j+1:3d}] = {lambda_qp[j]:12.5f}   {binding}")

        active_cstr = np.where(LAMBDA > 0)[0]
        if len(active_cstr) > 0:
            print(" * Active Constraints: " + "  ".join(f"{i+1:2d}" for i in active_cstr))
        print()

    # Trim history
    cvg_hist = cvg_hst[:, :iteration].copy()

    return v_opt, f_opt, g_opt, cvg_hist, lambda_qp, HESS


def solve_qp_subproblem(H, f, A, b):
    '''
    Solve QP subproblem using scipy.optimize.

    Solves: min 0.5*x'*H*x + f'*x  subject to  A*x <= b

    Parameters
    ----------
    H : np.ndarray (n, n)
        Hessian matrix
    f : np.ndarray (n,)
        Linear term gradient
    A : np.ndarray (m, n)
        Constraint matrix (inequality)
    b : np.ndarray (m,)
        Constraint RHS

    Returns
    -------
    x : np.ndarray (n,)
        Solution (search direction)
    lambda_vals : np.ndarray (m,)
        Lagrange multipliers (approximated from active constraints)
    how : str
        Status message
    '''
    n = H.shape[0]
    m = len(b)

    # Objective function: 0.5*x'*H*x + f'*x
    def objective(x):
        return 0.5 * np.dot(x, H.dot(x)) + np.dot(f, x)

    # Gradient of objective
    def grad_objective(x):
        return H.dot(x) + f

    # Inequality constraints: A*x - b <= 0
    constraints = []
    for i in range(m):
        constraints.append({
            'type': 'ineq',
            'fun': lambda x, i=i: b[i] - np.dot(A[i, :], x),
            'jac': lambda x, i=i: -A[i, :]
        })

    # Initial guess
    x0 = np.zeros(n)

    try:
        # Make H positive definite by adding small regularization if needed
        try:
            # Test if H is positive definite
            cho_factor(H + 1e-10 * np.eye(n))
            H_reg = H + 1e-10 * np.eye(n)
        except np.linalg.LinAlgError:
            # H is not positive definite, add more regularization
            H_reg = H + 1e-6 * np.eye(n)

        # Update objective with regularized H
        def objective_reg(x):
            return 0.5 * np.dot(x, H_reg.dot(x)) + np.dot(f, x)

        def grad_objective_reg(x):
            return H_reg.dot(x) + f

        # Solve with SLSQP
        result = minimize(
            objective_reg,
            x0,
            method='SLSQP',
            jac=grad_objective_reg,
            constraints=constraints,
            options={'maxiter': 200, 'ftol': 1e-9, 'disp': False}
        )

        if result.success:
            x = result.x
            how = 'ok'

            # Estimate Lagrange multipliers from active constraints
            lambda_vals = np.zeros(m)
            tol_active = 1e-6

            for i in range(m):
                constraint_val = b[i] - np.dot(A[i, :], x)
                if abs(constraint_val) < tol_active:  # Active constraint
                    # Approximate multiplier (should be from KKT conditions)
                    # For now, use a simple approximation
                    lambda_vals[i] = max(0, -np.dot(grad_objective_reg(x), A[i, :]))

        else:
            # Optimization failed
            x = result.x if hasattr(result, 'x') else x0
            lambda_vals = np.zeros(m)

            # Check what kind of failure
            if 'infeasible' in result.message.lower():
                how = 'infeasible'
            elif 'unbounded' in result.message.lower():
                how = 'unbounded'
            else:
                how = 'ill-posed'

        return x, lambda_vals, how

    except Exception as e:
        # Fallback: return zero solution
        x = np.zeros(n)
        lambda_vals = np.zeros(m)
        how = 'error: ' + str(e)[:20]
        return x, lambda_vals, how


"""
def solve_qp_subproblem(H, f, A, b):
    '''
    Solve QP using quadprog: min 0.5*x'*H*x + f'*x  s.t.  A*x <= b

    quadprog solves: min 0.5*x'*G*x - a'*x  s.t.  C'*x >= b

    Conversion:
    G = H
    a = -f
    C' = -A' ? C = -A.T
    b_qp = -b

    sudo pipx install quadprog --include-deps
    '''
    try:
        G = H
        a = -f
        C = -A.T
        b_qp = -b

        # solve_qp returns (solution, f_value, xu, iterations, lagrangian, iact)
        result = solve_qp(G, a, C, b_qp, meq=0)

        x = result[0]           # solution
        lagrangian = result[4]  # Lagrange multipliers

        how = 'ok'
        return x, lagrangian, how

    except ValueError as e:
        n = H.shape[0]
        x = np.zeros(n)
        lambda_vals = np.zeros(len(b))

        if 'infeasible' in str(e).lower():
            how = 'infeasible'
        elif 'unbounded' in str(e).lower():
            how = 'unbounded'
        else:
            how = 'ill-posed'

        return x, lambda_vals, how
"""

"\ndef solve_qp_subproblem(H, f, A, b):\n    '''\n    Solve QP using quadprog: min 0.5*x'*H*x + f'*x  s.t.  A*x <= b\n    \n    quadprog solves: min 0.5*x'*G*x - a'*x  s.t.  C'*x >= b\n    \n    Conversion:\n    G = H\n    a = -f\n    C' = -A' ? C = -A.T\n    b_qp = -b\n\n    sudo pipx install quadprog --include-deps\n    '''\n    try:\n        G = H\n        a = -f\n        C = -A.T\n        b_qp = -b\n        \n        # solve_qp returns (solution, f_value, xu, iterations, lagrangian, iact)\n        result = solve_qp(G, a, C, b_qp, meq=0)\n        \n        x = result[0]           # solution\n        lagrangian = result[4]  # Lagrange multipliers\n        \n        how = 'ok'\n        return x, lagrangian, how\n        \n    except ValueError as e:\n        n = H.shape[0]\n        x = np.zeros(n)\n        lambda_vals = np.zeros(len(b))\n        \n        if 'infeasible' in str(e).lower():\n            how = 'infeasible'\n        elif 'unbounded' in str(e).lower():\n            how = 

In [26]:

# --------------------------
# Problem data (direct translation)
# --------------------------
G = np.array([60.0, 225.0])
cA = 20.0
cB = 12.0

var_pairs = [
    ('A', 1), ('A', 2), ('A', 11), ('A', 12),
    ('B', 5), ('B', 6), ('B', 7), ('B', 8),
    (1, 2), (1, 3), (1, 12), (2, 3), (3, 4),
    (4, 5), (4, 6), (5, 6), (6, 7), (7, 8),
    (7, 9), (8, 9), (9, 10), (10, 11), (10, 12), (11, 12)
]
n = len(var_pairs)   # 24
m = 14               # 2 generator + 12 demand constraints

D = np.array([30., 20., 5., 7., 35., 25., 25., 20., 5., 6., 25., 30.])

# Build A (m x n)
A = np.zeros((m, n), dtype=float)
for idx, (i, j) in enumerate(var_pairs):
    if i == 'A':
        A[0, idx] = 1.0
    if i == 'B':
        A[1, idx] = 1.0

for node in range(1, 13):
    row = 2 + (node - 1)
    for var_idx, (i, j) in enumerate(var_pairs):
        if isinstance(i, int) and isinstance(j, int):
            if j == node and i < j:
                A[row, var_idx] = -1.0
            if i == node and i < j:
                A[row, var_idx] = +1.0
        else:
            if j == node and (i == 'A' or i == 'B'):
                A[row, var_idx] = -1.0
            if i == node and isinstance(j, int) and j > i:
                A[row, var_idx] = +1.0

# Build b
b = np.zeros((m,), dtype=float)
b[0] = G[0]
b[1] = G[1]
for node in range(1, 13):
    b[2 + (node - 1)] = -D[node - 1]

# Cost vector c
c = np.zeros((n,), dtype=float)
c[0:4] = cA
c[4:8] = cB

# Line capacities and bounds
T_val = 48.0
T = T_val * np.ones((n,), dtype=float)

x_lb = np.empty(n, dtype=float)
x_ub = np.empty(n, dtype=float)
for idx, (i, j) in enumerate(var_pairs):
    if i == 'A' or i == 'B':
        x_lb[idx] = 0.0
        x_ub[idx] = T[idx]
    else:
        x_lb[idx] = -T[idx]
        x_ub[idx] = +T[idx]

# Sanity check
sumA = A.sum(axis=0)
print("sum of A columns (sanity):", sumA)

# --------------------------
# Solve with linprog (HiGHS / dual-simplex preference)
# --------------------------
print("\nlinprog ----------------------")
A_ub = A.copy()
b_ub = b.copy()
bounds = [(float(x_lb[i]), float(x_ub[i])) for i in range(n)]

try:
    res = linprog(c=c, A_ub=A_ub, b_ub=b_ub, bounds=bounds, method='highs-ds')
    if not res.success:
        res = linprog(c=c, A_ub=A_ub, b_ub=b_ub, bounds=bounds, method='highs')
except Exception:
    res = linprog(c=c, A_ub=A_ub, b_ub=b_ub, bounds=bounds, method='highs')

x_opt = res.x
f_opt = res.fun
print("linprog x_opt:", x_opt)
print("linprog f_opt:", f_opt)

g_opt = A.dot(x_opt) - b
print("linprog g_opt (A*x - b):", g_opt)

y_opt = getattr(res, 'ineqlin', res)  # solver-dependent

netGeneration = np.sum(x_opt[0:4]) + np.sum(x_opt[4:8])
netDemand = np.sum(D)
print("netGeneration:", netGeneration)
print("netDemand:", netDemand)
print("shortfall (netDemand - netGeneration):", netDemand - netGeneration)

# --------------------------
# Solve with user's sqp solver
# --------------------------
print("\ncalling user's sqp solver ----------------------")

# constants as expected by LP_analysis
constants = (A, b, c)

# initial guess: midpoint of bounds
x_init = np.array([(lb + ub) / 2.0 for lb, ub in bounds])

# options vector similar to MATLAB: [msglev tolX tolF tolG MaxEvals]
options = [0, 0.01, 0.01, 0.01, 1e4]

# Directly call the user's sqp function (assumed available in namespace)
# Preferred call: sqp(function_callable, x_init, x_lb, x_ub, options, constants)
try:
    sqp_result = sqp(LP_analysis, x_init, x_lb, x_ub, options, constants)
except TypeError:
    # fallback: maybe sqp expects function name string instead of callable
    sqp_result = sqp('LP_analysis', x_init, x_lb, x_ub, options, constants)

# Unpack results (expecting tuple (x_opt, f_opt, g_opt, cvg_hst, y_opt))
if isinstance(sqp_result, tuple) and len(sqp_result) >= 3:
    x_opt_sqp = sqp_result[0]
    f_opt_sqp = sqp_result[1]
    g_opt_sqp = sqp_result[2]
    cvg_hst = sqp_result[3] if len(sqp_result) > 3 else None
    y_opt_sqp = sqp_result[4] if len(sqp_result) > 4 else None
else:
    # try attribute access if an object was returned
    x_opt_sqp = getattr(sqp_result, 'x', None)
    f_opt_sqp = getattr(sqp_result, 'fun', None)
    g_opt_sqp = getattr(sqp_result, 'g', None)
    cvg_hst = getattr(sqp_result, 'history', None)
    y_opt_sqp = getattr(sqp_result, 'y', None)

print("sqp x_opt:", x_opt_sqp)
print("sqp f_opt:", f_opt_sqp)
print("sqp g_opt (A*x - b):", g_opt_sqp)
print("sqp convergence/history:", cvg_hst)
print("sqp multipliers/y_opt:", y_opt_sqp)

netGeneration_sqp = np.sum(x_opt_sqp[0:4]) + np.sum(x_opt_sqp[4:8])
print("sqp netGeneration:", netGeneration_sqp)
print("sqp shortfall (netDemand - netGeneration):", netDemand - netGeneration_sqp)


sum of A columns (sanity): [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]

linprog ----------------------
linprog x_opt: [ 41.   0.   0.   0.  48.  48.  48.  48.  46. -48.  13.  26. -27.  14.
 -48.  27.   2.  20.   5.  48.  48.  48.  -6.  23.]
linprog f_opt: 3124.0
linprog g_opt (A*x - b): [-19. -33.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.]
netGeneration: 233.0
netDemand: 233.0
shortfall (netDemand - netGeneration): 0.0

calling user's sqp solver ----------------------
 * Woo Hoo!  Converged solution found in 3 iterations!
 *           convergence in design variables
 *           convergence in design objective
 * Woo Hoo!  Converged solution is feasible
sqp x_opt: [ 10.55528028   9.79640001  10.12667524  10.52164113  47.9999995
  47.9999995   47.9999995   47.9999995   -3.03552105 -16.27464126
  -0.13455657 -13.23912021 -34.51376147 -18.17125382 -23.34250764
  -5.17125382  -5.51376147  -3.50458716  20.99082569  24.49541284
  40.48623853  1

In [ ]:
from scipy.stats import norm

def plot_CDF_ci(data, confidence_level, figNo):
    """
    Plot empirical CDF of data with confidence intervals.

    Parameters
    ----------
    data : array_like
        Data values
    confidence_level : float
        Confidence level (e.g., 95 for 95%)
    figNo : int
        Figure number
    """

    data = np.asarray(data).flatten()
    n = len(data)

    # Sort data
    sorted_data = np.sort(data)

    # Empirical CDF
    ecdf = np.arange(1, n + 1) / n

    # Confidence intervals using Dvoretzky-Kiefer-Wolfowitz inequality
    alpha = 1 - confidence_level / 100
    epsilon = np.sqrt(np.log(2 / alpha) / (2 * n))

    upper_ci = np.minimum(ecdf + epsilon, 1.0)
    lower_ci = np.maximum(ecdf - epsilon, 0.0)

    # Theoretical normal CDF
    mu, std = np.mean(data), np.std(data)
    x_theory = np.linspace(min(sorted_data), max(sorted_data), 200)
    cdf_theory = norm.cdf(x_theory, mu, std)

    # Plotting
    fig = plt.figure(figNo, figsize=(10, 7))
    fig.clf()

    plt.fill_between(sorted_data, lower_ci, upper_ci,
                     color=[0.8, 0.9, 1.0], alpha=0.5,
                     label=f'{confidence_level}% confidence band')
    plt.plot(sorted_data, ecdf, 'o-', color=[0.2, 0.4, 0.8],
             linewidth=2, markersize=4, label='Empirical CDF')
    plt.plot(x_theory, cdf_theory, 'r-', linewidth=2,
             label=f'Normal CDF (μ={mu:.3f}, σ={std:.3f})')
    plt.plot(sorted_data, sorted_data * 0 + 0.5, '--k',
             alpha=0.3, linewidth=1)

    plt.xlabel('Residual value', fontsize=13)
    plt.ylabel('Cumulative probability', fontsize=13)
    plt.title(f'Empirical CDF with {confidence_level}% Confidence Intervals',
              fontsize=14)
    plt.legend(fontsize=11)
    plt.grid(True, alpha=0.3)
    plt.ylim([0, 1])

    plt.tight_layout()